## 2.1 Imputing, Encoding and First Pipeline

In [2]:
from pandas.core.common import random_state
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, classification_report, f1_score, accuracy_score
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from features import *
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


data = pd.read_csv('train.csv')
child_feature(data)
family_size_feature(data)
name_feature(data)
group_rare_title(data)

In [3]:
X = data.drop(columns=['Survived', 'Ticket', 'PassengerId', 'Cabin', 'Name'])
y = data['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Child,FamilySize,Title
0,3,male,22.0,1,0,7.2500,S,0,2,Mr
1,1,female,38.0,1,0,71.2833,C,0,2,Mrs
2,3,female,26.0,0,0,7.9250,S,0,1,Miss
3,1,female,35.0,1,0,53.1000,S,0,2,Mrs
4,3,male,35.0,0,0,8.0500,S,0,1,Mr


### Categorical and numerical features

In [4]:
cat_features = list(X.drop(columns=X.select_dtypes(include=['int', 'float'])))
num_features = list(X.drop(columns=X.select_dtypes(include=['str'])))

### Numerical and categorical pipeline + Imputer

In [5]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

### ColumnTransfer

In [6]:
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

### Final Pipeline

In [7]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(C=1.0, max_iter=1000))
])

final_pipeline.fit(X_train, y_train)
prediction = final_pipeline.predict(X_test)

### Confusion Matrix, predict_proba, precision, recall

In [8]:
conf_matrix = confusion_matrix(y_test, prediction)
print(conf_matrix)

proba_survived = final_pipeline.predict_proba(X_test)[:, 1]
new_labels = (proba_survived >= 0.7).astype(int)

new_conf_matrix = confusion_matrix(y_test, new_labels)
print(new_conf_matrix)

[[89 16]
 [21 53]]
[[96  9]
 [34 40]]


### Important to mention:
 - 1. We have to increase recall because it is better to indentify person as dead and he is not than identify person as alive and he is dead

### Cross Validation

In [9]:
cross_validation = cross_val_score(final_pipeline, X, y, cv=5)
print(np.mean(cross_validation))

crv_predict = cross_val_predict(final_pipeline, X, y, cv=5)
conf_matrix = confusion_matrix(y, crv_predict)
print(conf_matrix)

class_report = classification_report(y, crv_predict)
print(class_report)

0.8215491808423827
[[480  69]
 [ 90 252]]
              precision    recall  f1-score   support

           0       0.84      0.87      0.86       549
           1       0.79      0.74      0.76       342

    accuracy                           0.82       891
   macro avg       0.81      0.81      0.81       891
weighted avg       0.82      0.82      0.82       891



### Class imbalance affects performance

 According to classification report we can see that results from the 0 class (death people) is higher than 1 class (survived)acros all three metrics - precision (0.83 vs 0.76), recall (0.86 vs 0.71), f1-score (0.84 vs 0.73). This is caused by more instances from class 0 (549 vs 342).

### DecisionTree vs RandomForest vs LogReg

In [10]:
models = [
    ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
    ('DecisionTreeClassifier', DecisionTreeClassifier(random_state=42)),
    ('RandomForestClassifier', RandomForestClassifier(random_state=42)),
]

results = []

for name, model in models:
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    cross_predict = cross_val_predict(pipeline, X, y, cv=5)
    precision = precision_score(y, cross_predict, average=None)
    recall = recall_score(y, cross_predict, average=None)
    f1 = f1_score(y, cross_predict, average=None)
    model_results = {
        'model': name,
        'accuracy': accuracy_score(y, cross_predict),
        'precision_dead': precision[0],
        'precision_survived': precision[1],
        'recall_dead': recall[0],
        'recall_survived': recall[1],
        'f1_dead': f1[0],
        'f1_survived': f1[1],
    }
    results.append(model_results)

### GridSearch

In [11]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42)),
])

param_grid = {
    'model__n_estimators': [200, 220, 250, 270],
    'model__criterion': ['gini', 'entropy'],
    'model__max_depth': [4, 5, 6, 7],
    'model__min_samples_split': [10, 12, 14, 16],
    'model__min_samples_leaf': [4, 5, 6],
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
precision = precision_score(y_test, y_pred, average=None)
recall = recall_score(y_test, y_pred, average=None)
f1 = f1_score(y_test, y_pred, average=None)

new_results = {
    'model': f"Random Forest (Best: {grid_search.best_params_})",
    'accuracy': accuracy_score(y_test, y_pred),
    'precision_dead': precision[0],
    'precision_survived': precision[1],
    'recall_dead': recall[0],
    'recall_survived': recall[1],
    'f1_dead': f1[0],
    'f1_survived': f1[1],
}

results.append(new_results)
df_results = pd.DataFrame(results)
df_results.head()

,model,accuracy,precision_dead,precision_survived,recall_dead,recall_survived,f1_dead,f1_survived
0,LogisticRegression,0.821549,0.842105,0.785047,0.874317,0.736842,0.857909,0.760181
1,DecisionTreeClassifier,0.767677,0.813187,0.695652,0.808743,0.701754,0.810959,0.698690
2,RandomForestClassifier,0.806958,0.837209,0.756024,0.852459,0.733918,0.844765,0.744807
3,Random Forest (Best: {'model__criterion': 'gin...,0.821229,0.828829,0.808824,0.876190,0.743243,0.851852,0.774648


In [12]:
print(grid_search.best_score_)
print(f1_score(y_test, y_pred, average='weighted'))

0.8353751207885066
0.8199351290861244


### Best Model

In [24]:
clean_params = {k.replace('model__', ''): v for k, v in grid_search.best_params_.items()}

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(**clean_params, random_state=42)),
])
final_model.fit(X, y)

test_data = pd.read_csv("test.csv")
child_feature(test_data)
family_size_feature(test_data)
name_feature(test_data)
group_rare_title(test_data)

passenger_id = test_data['PassengerId']
test_data = test_data.drop(columns=['PassengerId', 'Ticket', 'Name', 'Cabin'])

predictions = final_model.predict(test_data)

submission = {
    'PassengerId': passenger_id,
    'Survived': predictions,
}

submission_data_frame = pd.DataFrame(submission)
submission_data_frame.to_csv("submission.csv", index=False)

### Kaggle Submission:
 - Firts Model RandomForest with GridSearch - 0.77751